# Pretrained v1: an off-the-shelf base model

Trains torchvision's ResNet-18 on the project's 64x64 prefix images, from ImageNet weights and from scratch,
two seeds each (`configs/pretrained_v1.yaml`), with exactly the protocol of the project's CNN (`cnn_prefix`).
Then scores them on when to guess and on the class tree, against `cnn_prefix` from its saved predictions.
**Validation data only:** the test split was opened once in final_v1 and is not opened here. Every comparison
was fixed beforehand.

**Before running:**
1. In **Input**: your Kaggle Dataset with the processed data, and the dataset `ai-pictionary-step4a-output`
   (version 4, which holds `runs/predictions_all/*.npz`, including `cnn_prefix_s42.npz` and `cnn_prefix_s43.npz`).
   The notebook stops early if it cannot find them.
2. **Internet on**: needed for the code and for the ImageNet weights.
3. Choose **GPU T4 x2**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed: stopping before using GPU quota")
mine = subprocess.run("PYTHONPATH=src python -m pytest tests/test_pretrained.py -q -rs", shell=True, capture_output=True, text=True)
if mine.returncode != 0 or "SKIPPED" in mine.stdout:
    print(mine.stdout[-1500:])
    raise RuntimeError("the ResNet-18 tests failed or were skipped (torch or torchvision missing): stopping")

import numpy as np

# Saved VALIDATION predictions of cnn_prefix: a folder named predictions_all (never the test predictions of final_v1).
OLD_PREDICTIONS = None
for p in Path("/kaggle/input").rglob("cnn_prefix_s42.npz"):
    if p.parent.name == "predictions_all" and (p.parent / "cnn_prefix_s43.npz").is_file():
        OLD_PREDICTIONS = p.parent
        break
if OLD_PREDICTIONS is None:
    raise RuntimeError("runs/predictions_all with cnn_prefix_s42/s43.npz not found: attach ai-pictionary-step4a-output (version 4)")
for seed in (42, 43):
    halves = set(np.unique(np.load(OLD_PREDICTIONS / f"cnn_prefix_s{seed}.npz")["half"]).tolist())
    if halves != {0, 1}:
        raise RuntimeError(f"cnn_prefix_s{seed}.npz is not a validation prediction file (halves {halves})")
print("saved validation predictions:", OLD_PREDICTIONS)


In [ ]:
# Download the ImageNet weights once, before two training jobs start in parallel.
import torch, torchvision
from torchvision.models import ResNet18_Weights, resnet18
print("torch", torch.__version__, "torchvision", torchvision.__version__)
try:
    resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
except Exception as e:
    raise RuntimeError("could not download the ImageNet weights: turn Internet on in the notebook settings") from e
print("ImageNet weights ready")


In [ ]:
MAX_MINUTES = 480
TOP = Path("/kaggle/working/runs/pretrained_v1")
!PYTHONPATH=src python -m quickdraw_exp.experiments --matrix configs/pretrained_v1.yaml --max-minutes {MAX_MINUTES}
# Checkpoint predictions of the new models on the validation halves (GPU). The summary printed here lacks
# cnn_prefix: that is expected, it is added in the next cell.
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/pretrained_v1.yaml --runs {TOP} --out {TOP}/predictions --num-workers 4


In [ ]:
# The predictions needed here in one folder, kept in this version's output for later steps.
ALL = Path("/kaggle/working/runs/predictions_pretrained")
ALL.mkdir(parents=True, exist_ok=True)
for p in [OLD_PREDICTIONS / "cnn_prefix_s42.npz", OLD_PREDICTIONS / "cnn_prefix_s43.npz", *(TOP / "predictions").glob("*.npz")]:
    shutil.copy(p, ALL / p.name)
print(sorted(p.name for p in ALL.glob("*.npz")))
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/pretrained_v1.yaml --predictions {ALL} --out /kaggle/working/runs/pretrained_v1_anytime
!PYTHONPATH=src python -m quickdraw_exp.hierarchy --matrix configs/hierarchy_eval_pretrained_v1.yaml --predictions {ALL} --out /kaggle/working/runs/hierarchy_pretrained_v1


In [ ]:
import json

training = json.loads((TOP / "results.json").read_text())
print("training", "COMPLETE" if training["complete"] else "INCOMPLETE, missing: " + ", ".join(training["missing"]))
leak = json.loads(Path("reports/leak_v1/results.json").read_text())["table"]
print("\nTop-1 by ink fraction, val 1000 per class (cnn_prefix from reports/leak_v1):")
for name, row in [("cnn_prefix", leak["cnn_prefix"]), *training["table"].items()]:
    print(f"  {name:22s} seeds {row['seeds']}", {f: round(v["top1"]["mean"], 4) for f, v in row["by_frac"].items()},
          "early", round(row["early_top1"]["mean"], 4))
for name in sorted(training["table"]):
    for run in sorted(TOP.glob(f"{name}_s*/summary.json")):
        s = json.loads(run.read_text())
        print(f"  {run.parent.name}: parameters {s.get('parameters')}, steps {s.get('steps')}, "
              f"{s.get('train_samples_per_s')} samples/s")

result = json.loads(Path("/kaggle/working/runs/pretrained_v1_anytime/results.json").read_text())
print("\nmissing conditions:", result["missing_conditions"])
print("\nPre-registered comparisons (AUC, evaluation half):")
for row in result["comparisons"]:
    if row["difference"] is None:
        print(f"  {row['name']}: not available")
    else:
        print(f"  {row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, difference {row['difference']:+.4f}  "
              f"(a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")
print("\nRisk control and AUC per condition:")
for condition, policies in result["table"].items():
    for policy, e in policies.items():
        if "earliness_at_gap_0.02" in e:
            print(f"  {condition:22s} {policy:7s} seeds {e['seeds']} auc {e['auc']['mean']:.4f}  earliness@gap5% "
                  f"{e['earliness_at_gap_0.05']['mean']:.3f} met {e['met_gap_0.05']}")
for run, ref in result["reference"].items():
    print(f"  {run}: end of drawing {ref['end_of_drawing']['accuracy']:.4f}")

hier = json.loads(Path("/kaggle/working/runs/hierarchy_pretrained_v1/results.json").read_text())
print("\nmissing runs (hierarchy):", hier["missing_runs"])
print("Hierarchy comparisons (reward area, evaluation half):")
for row in hier["comparisons"]:
    if row["difference"] is None:
        print(f"  {row['name']} @ {row['precision_target']}: not available")
    else:
        print(f"  {row['name']} @ {row['precision_target']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, "
              f"difference {row['difference']:+.4f}")
for name in training["missing"]:
    log = TOP / f"{name}.log"
    if log.exists():
        print(f"--- {name} ---")
        print("\n".join(log.read_text().splitlines()[-15:]))
